In [1]:
!pip install transformers
!pip install datasets
!pip install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 18.8 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which

In [2]:
import pandas as pd
from datasets import Dataset
from transformers import GPT2Tokenizer

In [3]:
# Load the dataset
train_path = "/content/train.csv"
valid_path = "/content/valid.csv"
train_df = pd.read_csv(train_path)
valid_df = pd.read_csv(valid_path)

In [4]:
print(train_df.head())
print(valid_df.head())

                                             answers  \
0  Kids who are bipolar, in their manic stages, v...   
1                  Equifax, transunion and experian.   
2  Women eat at least 1,200 calories daily and me...   
3  Because Caffeine increases the stress hormone ...   
4                                        Kent County   

                                               query  \
0                     why do children get aggressive   
1  which credit bureau is used the most for auto ...   
2         what is the minimum healthy calorie intake   
3                   why is coffee making gain weight   
4                 what county is grand rapids, mi in   

                                        finalpassage  
0  At the same time, despite claiming the review ...  
1  Best Answer: both of those answers are wrong. ...  
2  Safe Intakes. If you’re not supervised by a me...  
3  Is coffee making you fat? If you are overweigh...  
4  Located in Grand Rapids, Michigan, the 61st Di..

In [5]:
train_df.columns
valid_df.columns

Index(['answers', 'query', 'finalpassage'], dtype='object')

In [6]:
print(train_df.isnull().sum())
print(valid_df.isnull().sum())

answers         55
query            0
finalpassage    24
dtype: int64
answers         4
query           0
finalpassage    1
dtype: int64


In [7]:
train_df = train_df.dropna(subset=['query', 'answers', 'finalpassage'])
valid_df = valid_df.dropna(subset=['query', 'answers', 'finalpassage'])

In [8]:
print(train_df.isnull().sum())
print(valid_df.isnull().sum())

answers         0
query           0
finalpassage    0
dtype: int64
answers         0
query           0
finalpassage    0
dtype: int64


In [9]:
# Example of Few-Shot Prompt
def generate_few_shot_prompt(query, answer, examples=3):
    """Generate a few-shot learning prompt."""
    prompt = ""
    for i in range(examples):
        prompt += f"Q: What is {train_df.iloc[i]['query']}?\nA: {train_df.iloc[i]['answers']}\n\n"
    prompt += f"Q: {query}\nA: "
    return prompt

In [10]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Load GPT-2 model and tokenizer
model = GPT2LMHeadModel.from_pretrained("gpt2")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# Define function for generating response
def generate_response(prompt, max_tokens=150):
    """Generate response using GPT-2 model with Few-shot or CoT prompt."""
    # Tokenize the prompt
    inputs = tokenizer.encode(prompt, return_tensors="pt")

    # Generate output (response)
    outputs = model.generate(
        inputs,
        max_new_tokens=max_tokens,  # Controls the maximum number of tokens to generate
        num_return_sequences=1,
        no_repeat_ngram_size=2,
        pad_token_id=50256  # Using eos_token as pad_token
    )

    # Decode the generated output
    decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded_output

# Example query to test the model
example_query = "What is the capital of France?"
prompt = generate_few_shot_prompt(example_query, None, examples=3)
response = generate_response(prompt)

print(f"Generated Response: {response}")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Generated Response: Q: What is why do children get aggressive?
A: Kids who are bipolar, in their manic stages, very frequently become aggressive. They lose self-control, they become impulsive. On the other end of the spectrum, when they become depressed, although aggression is less common, they can become irritable, and sometimes that irritability and cantankerousness causes kids to lash out.

Q: What is which credit bureau is used the most for auto loans?
A: Equifax, transunion and experian.

Q: What is what is the minimum healthy calorie intake?
A: Women eat at least 1,200 calories daily and men get a minimum of 1,500 calories a day.

Q: What is the capital of France?
A: Â France is a country of about 1.5 million people. It is home to about 2.4 million inhabitants. The capital is Paris.


In [11]:
!pip install rouge-score
from sklearn.metrics import f1_score
from rouge_score import rouge_scorer

def compute_rouge_l(predictions, references):
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    scores = [scorer.score(ref, pred) for ref, pred in zip(references, predictions)]
    return sum([score['rougeL'].fmeasure for score in scores]) / len(scores)

# Evaluate BERT F1 score
def compute_bert_f1(predictions, references):
    return f1_score(references, predictions, average='macro')

# Sample predictions and references (use actual predictions for real evaluation)
predictions = ['Paris', 'blue whale', 'Yen', 'Paris']  # Example model outputs
references = ['Paris', 'blue whale', 'Yen', 'Paris']   # Ground truth answers

rouge_l_score = compute_rouge_l(predictions, references)
f1_score_val = compute_bert_f1(predictions, references)

print(f"ROUGE-L Score: {rouge_l_score}")
print(f"BERT-F1 Score: {f1_score_val}")


  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=bdf1d0640c5ef2f6724b8462f25a5aef952d609c156d2e26dccf97dc6462255c
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge-score
ROUGE-L Score: 1.0
BERT-F1 Score: 1.0


In [12]:
from datasets import Dataset
from transformers import Trainer, TrainingArguments, GPT2LMHeadModel, DataCollatorForLanguageModeling, GPT2Tokenizer

# Combine 'query' and 'finalpassage' to form the input prompt
train_df['input'] = train_df['query'] + " " + train_df['finalpassage']
valid_df['input'] = valid_df['query'] + " " + valid_df['finalpassage']

# Prepare target output (answer)
train_df['output'] = train_df['answers']
valid_df['output'] = valid_df['answers']

# Convert to Hugging Face dataset format
train_dataset = Dataset.from_pandas(train_df[['input', 'output']])
valid_dataset = Dataset.from_pandas(valid_df[['input', 'output']])

# Load GPT-2 tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # Ensure padding is consistent

# Tokenization function with padding mask fix
def tokenize_function(examples):
    model_inputs = tokenizer(
        examples['input'], truncation=True, padding='max_length', max_length=128
    )
    labels = tokenizer(
        examples['output'], truncation=True, padding='max_length', max_length=128
    )

    # Replace padding token in labels with -100 (ignored by loss function)
    labels["input_ids"] = [
        [(token if token != tokenizer.pad_token_id else -100) for token in label]
        for label in labels["input_ids"]
    ]

    model_inputs['labels'] = labels['input_ids']
    return model_inputs

# Apply tokenization
train_dataset = train_dataset.map(tokenize_function, batched=True)
valid_dataset = valid_dataset.map(tokenize_function, batched=True)

# Rename columns for input and labels
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
valid_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# Load GPT-2 model
model = GPT2LMHeadModel.from_pretrained("gpt2")

# Data collator for language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # GPT-2 does not use masked language modeling
)

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",          # Output directory for model checkpoints
    evaluation_strategy="epoch",     # Evaluate after every epoch
    save_strategy="epoch",           # Save checkpoints after every epoch
    logging_dir="./logs",            # Directory for logging
    num_train_epochs=3,              # Number of epochs
    per_device_train_batch_size=8,   # Batch size for training
    per_device_eval_batch_size=8,    # Batch size for evaluation
    logging_steps=10,                # Log every 10 steps
    save_steps=500,                  # Save checkpoint every 500 steps
    load_best_model_at_end=True,     # Load the best model at the end of training
    report_to="none",                 # Disable W&B logging
    save_total_limit=2                # Keep only the last 2 checkpoints
)

# Create the Trainer object
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    tokenizer=tokenizer,  # This warning can be ignored for now
)

# Start training the model
trainer.train()


Map:   0%|          | 0/119921 [00:00<?, ? examples/s]

Map:   0%|          | 0/10580 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-12-4e03dcafbd9d>:72: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.022100,2.972651
2,3.051100,2.926144
3,2.772000,2.911009


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=44973, training_loss=2.9687585291645027, metrics={'train_runtime': 3223.2376, 'train_samples_per_second': 111.615, 'train_steps_per_second': 13.953, 'total_flos': 2.3500801327104e+16, 'train_loss': 2.9687585291645027, 'epoch': 3.0})

In [27]:
trainer.save_model("fine_tuned_gpt2")

In [31]:
# Inspect one example from the dataset to check the correct key for target labels
print(valid_dataset[0])

{'input_ids': tensor([ 4919,  2968,   318,   262,  1438,   369,  6335,   383,  1438, 39708,
          318,   257,  5156,  2933,  1438,    13,   383,  1438, 39708,   318,
          257,  2679,  5156,  1438,    13,   554,  2679,   262,  3616,   286,
          262,  1438, 39708,   318,    25, 28106, 20685, 22813,    26, 10787,
           13,   383,  1438, 39708,   318,   281,  1605,  5156,  1438,    13,
          554,  1605,   262,  3616,   286,   262,  1438, 39708,   318,    25,
        28106, 20685, 22813,    26, 10787,    13,   383,  1438, 39708,   318,
          257, 34314,   291,  5156,  1438,    13,   554, 34314,   291,   262,
         3616,   286,   262,  1438, 39708,   318,    25, 22813,    26, 10787,
           13,   383,  1438, 39708,   318,   257,  1665,   315,  9229,  5156,
         1438,    13,  3103,  6335,   318,   257,   845,  2968,   717,  1438,
          329,  1450, 17426, 29228,   503,   286,  1105,  1238,    11,  5849,
         5214,  4407,   290,   635,   257,   845, 

In [43]:
from rouge_score import rouge_scorer

def compute_rouge_l(predictions, references):
    # Initialize the ROUGE scorer for ROUGE-L
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

    # Compute ROUGE-L for each pair of prediction and reference
    scores = [scorer.score(reference, prediction)['rougeL'] for prediction, reference in zip(predictions, references)]

    # Compute average ROUGE-L scores
    avg_precision = sum([score.precision for score in scores]) / len(scores)
    avg_recall = sum([score.recall for score in scores]) / len(scores)
    avg_f1 = sum([score.fmeasure for score in scores]) / len(scores)

    return avg_precision, avg_recall, avg_f1

# Example usage:
predictions = ["This is a test sentence.", "Hello world!"]
references = ["This is a test.", "Hello, world!"]
avg_precision, avg_recall, avg_f1 = compute_rouge_l(predictions, references)

print(f"ROUGE-L Precision: {avg_precision:.4f}")
print(f"ROUGE-L Recall: {avg_recall:.4f}")
print(f"ROUGE-L F1: {avg_f1:.4f}")

ROUGE-L Precision: 0.9000
ROUGE-L Recall: 1.0000
ROUGE-L F1: 0.9444


In [47]:
from bert_score import score

def compute_bert_score(predictions, references):
    P, R, F1 = score(predictions, references, lang="en", model_type="bert-base-uncased")
    return P.mean().item(), R.mean().item(), F1.mean().item()

# Example usage
predictions = ["This is a test sentence.", "Hello world!"]
references = ["This is a test.", "Hello, world!"]

bert_precision, bert_recall, bert_f1 = compute_bert_score(predictions, references)

print(f"BERT Precision: {bert_precision:.4f}")
print(f"BERT Recall: {bert_recall:.4f}")
print(f"BERT F1 Score: {bert_f1:.4f}")

BERT Precision: 0.8585
BERT Recall: 0.8464
BERT F1 Score: 0.8503
